# Fine-tuning do Assistente Medico - Hospital Pos Tech
Tech Challenge Fase 3 (Generative AI)

Este notebook faz o fine-tuning (LoRA, via Unsloth) de um LLM open-source
sobre o dataset clinico interno do hospital (`data/processed/dataset_fine_tuning.jsonl`),
seguindo o mesmo padrao do material de referencia do curso
(`fine-tuning-rag-documentos-fiap-main/Aula 02 - Fine tuning de LLM para documentos`).

**Requer GPU** (rode no Google Colab com ambiente de execucao GPU, ou em uma
maquina com GPU NVIDIA). Para uma demonstracao que roda sem GPU, veja
`src/fine_tuning/train_demo_cpu.py`.

Passos: (1) instalar dependencias, (2) montar o Google Drive e subir o
projeto, (3) carregar o dataset, (4) carregar o modelo base em 4-bit,
(5) aplicar LoRA, (6) treinar com SFTTrainer, (7) testar inferencia,
(8) salvar o adaptador em `models/fine_tuned_lora/`.

In [ ]:
#Conexao com o Google Drive (rode no Colab)
from google.colab import drive
drive.mount('/content/drive')

Suba a pasta `tech_challenge` para o seu Google Drive (ex.: em
`MyDrive/tech_challenge`) e ajuste `PROJECT_DIR` abaixo.

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/tech_challenge"  # ajuste se necessario

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers peft accelerate bitsandbytes
!pip install --upgrade transformers trl datasets
# Garante a versao mais recente do Unsloth (a instalacao acima pode reaproveitar
# um cache antigo) - recomendacao oficial do Unsloth para evitar incompatibilidade
# de versao com trl/transformers (ver docs.unsloth.ai/basics/troubleshooting-and-faqs).
!pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
import json
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

DATASET_PATH = f"{PROJECT_DIR}/data/processed/dataset_fine_tuning.jsonl"
OUTPUT_MODEL_DIR = f"{PROJECT_DIR}/models/fine_tuned_lora"

max_seq_length = 2048
dtype = None
load_in_4bit = True

# Mesma familia de modelos usada no material de referencia do curso.
model_name = "unsloth/llama-3-8b-bnb-4bit"


## Carregando o dataset

O dataset ja esta no formato `instruction` / `input` / `output`
(ver `src/data_prep/build_fine_tuning_dataset.py`), o mesmo esperado pelo
prompt Alpaca usado no material de referencia.

In [ ]:
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Exemplos no dataset: {len(dataset)}")
dataset[0]

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = None  # definido apos carregar o tokenizer

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input_, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

EOS_TOKEN = tokenizer.eos_token
dataset = dataset.map(formatting_prompts_func, batched=True)

## Aplicando LoRA

Mesmos hiperparametros do material de referencia (r=16, alpha=16), que sao
um bom ponto de partida para datasets pequenos/medios como o nosso.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        dataset_num_proc=2,
        packing=False,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,   # dataset pequeno: repetir algumas epocas
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

In [ ]:
trainer_stats = trainer.train()

## Testando a inferencia com um caso clinico de exemplo

In [ ]:
FastLanguageModel.for_inference(model)

pergunta_teste = "Qual o protocolo interno para Diabetes Mellitus tipo 2?"

inputs = tokenizer(
    [alpaca_prompt.format(
        "Responda como assistente clinico do hospital, com base no protocolo interno.",
        pergunta_teste,
        "",
    )],
    return_tensors="pt",
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=200, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])

## Salvando o adaptador LoRA

O `src/models/domain_llm.py` carrega o modelo a partir deste caminho quando
`FINE_TUNED_MODEL_PATH` (no `.env`) apontar para ele.

In [ ]:
model.save_pretrained(OUTPUT_MODEL_DIR)
tokenizer.save_pretrained(OUTPUT_MODEL_DIR)
print(f"Adaptador salvo em {OUTPUT_MODEL_DIR}")

## Avaliacao do modelo

Para o relatorio tecnico do desafio, compare respostas do modelo base vs.
fine-tuned para as mesmas perguntas do conjunto de teste, e registre:
- aderencia ao protocolo interno (avaliacao qualitativa manual);
- perplexidade no conjunto de validacao;
- ROUGE-L entre a resposta gerada e a resposta de referencia.

Um esqueleto de avaliacao fica em `src/fine_tuning/evaluate.py`.